# TextRCNN with SGNS

Use the tuned SGNS training configuration and select the checkpoint by validation macro-F1.

Run from the repository root.

# TextRCNN + sgns — square-root class weighting
Place the archived model inputs in data_64_16_20 before running this notebook from top to bottom.
Review the configuration before running the training cells.
Fit embeddings and the classifier on training data. Select the checkpoint on validation and enable test evaluation only after fixing the configuration.
Use the same data splits and evaluation protocol when comparing model configurations.


In [ ]:
%pip install torch

In [ ]:
%pip install scikit-learn

In [ ]:
# Independent model program: no imports from the old experiment workspace.
import argparse, hashlib, json, random, time, zlib
from pathlib import Path
from collections import Counter
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report, confusion_matrix

MODEL_NAME = 'TextRCNN'
EMBEDDING_METHOD = 'sgns'
# ===== Model configuration =====
SEED = 42
DIMENSION = 64          # Increase the embedding dimension from 48 to 64.
MAX_LENGTH = 128        # Increase the review limit from 96 to 128 tokens.
WINDOW = 5              # Expand the context window from 4 to 5.
BUCKETS = 50000         
EPOCHS = 15             # Increase the maximum training epochs from 5 to 15.
EMBEDDING_EPOCHS = 8    # Increase embedding training epochs from 2 to 8.
BATCH_SIZE = 128
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.02     # Increase weight decay from 0.01 to 0.02.
GRADIENT_CLIP = 5
MIN_COUNT = 2
MAX_VOCAB = 30000
CHANNELS = 128          # Increase the projection dimension from 64 to 128.
BLOCKS = 6             
KERNEL_SIZE = 3        
RNN_HIDDEN = 64         # Increase the recurrent hidden dimension from 32 to 64.
WEIGHT_ALPHA = 0.5      
# ===== End of configuration =====
NAMES = ['low', 'moderate', 'good', 'excellent']
PACKAGE_DIR = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()



In [ ]:
def character_ngrams(word):
    bounded = f'<{word}>'
    return [1 + zlib.crc32(bounded[i:i+n].encode()) % BUCKETS
            for n in range(3, 7) for i in range(len(bounded) - n + 1)
            if n != len(bounded)]

def component_table(words):
    rows = [[]] + [[BUCKETS + index, *character_ngrams(word)]
                   for index, word in enumerate(words, start=1)]
    table = torch.zeros((len(rows), max(map(len, rows))), dtype=torch.long)
    for index, row in enumerate(rows):
        table[index, :len(row)] = torch.tensor(row, dtype=torch.long)
    return table

def make_pairs(corpus):
    counts = Counter()
    for sentence in corpus:
        for i, center in enumerate(sentence):
            if center == 0:
                continue
            for j in range(max(0, i - WINDOW), min(len(sentence), i + WINDOW + 1)):
                if i != j and sentence[j] != 0:
                    counts[center, sentence[j]] += 1.0 / abs(i - j)
    pairs = np.asarray(list(counts), dtype=np.int64)
    values = np.fromiter(counts.values(), dtype=np.float32, count=len(counts))
    return torch.from_numpy(pairs), torch.from_numpy(values)

def train_embedding(method, words, pairs, counts, frequency, oov, epochs):
    torch.manual_seed(SEED)
    size = len(words) + 1
    subwords = component_table(words) if method == 'fasttext' else None
    input_size = size + BUCKETS if subwords is not None else size
    source = nn.Embedding(input_size, DIMENSION, padding_idx=0, sparse=True)
    context = nn.Embedding(size, DIMENSION, padding_idx=0, sparse=True)
    nn.init.uniform_(source.weight[1:], -0.5 / DIMENSION, 0.5 / DIMENSION)
    if method == 'glove':
        nn.init.uniform_(context.weight[1:], -0.5 / DIMENSION, 0.5 / DIMENSION)
        source_bias, context_bias = nn.Embedding(size, 1, sparse=True), nn.Embedding(size, 1, sparse=True)
        nn.init.zeros_(source_bias.weight)
        nn.init.zeros_(context_bias.weight)
        parameters = [*source.parameters(), *context.parameters(), *source_bias.parameters(), *context_bias.parameters()]
    else:
        nn.init.zeros_(context.weight)
        parameters = [*source.parameters(), *context.parameters()]
    optimizer = torch.optim.SparseAdam(parameters, lr=0.025 if method == 'glove' else 0.01)
    noise = frequency.pow(0.75)

    def centers(ids):
        if subwords is None:
            return source(ids)
        components = subwords[ids]
        return source(components).sum(dim=1) / components.ne(0).sum(dim=1, keepdim=True).clamp_min(1)

    for epoch in range(epochs):
        if method == 'glove':
            order = torch.randperm(len(pairs))
        else:
            # Sample observed pairs by distance-weighted frequency for SGNS.
            order = torch.multinomial(counts, len(pairs), replacement=True)
        loss_sum = 0.0
        for batch in order.split(4096):
            left, right = pairs[batch].T
            vectors = centers(left)
            positive = (vectors * context(right)).sum(dim=1)
            if method == 'glove':
                weights = counts[batch]
                residual = positive + source_bias(left).flatten() + context_bias(right).flatten() - weights.log()
                loss = (((weights / 10).clamp(max=1).pow(0.75)) * residual.square()).mean()
            else:
                negatives = torch.multinomial(noise, len(batch) * 5, replacement=True).reshape(-1, 5)
                negative = torch.einsum('bd,bkd->bk', vectors, context(negatives))
                loss = -(F.logsigmoid(positive) + F.logsigmoid(-negative).sum(dim=1)).mean()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(batch)
        print(f'{method} embedding epoch {epoch + 1}/{epochs} loss={loss_sum / len(order):.4f}', flush=True)

    with torch.no_grad():
        vectors = torch.cat([centers(ids) for ids in torch.arange(size).split(1024)])
        if method == 'glove':
            vectors += context.weight
        if subwords is not None:
            # Ignore hash buckets absent from the training vocabulary.
            used = set(subwords.flatten().tolist())
            for word in oov:
                pieces = [piece for piece in character_ngrams(word) if piece in used]
                oov[word] = source.weight[pieces].mean(dim=0) if pieces else torch.zeros(DIMENSION)
        return F.normalize(vectors, dim=1), (source.weight.detach().clone() if subwords is not None else None), (sorted(used) if subwords is not None else [])

def encode(rows, vocabulary, unknown_id):
    tokens = torch.zeros((len(rows), MAX_LENGTH), dtype=torch.long)
    labels = torch.tensor([label for _, label in rows], dtype=torch.long)
    for index, (review, _) in enumerate(rows):
        selected = review[:MAX_LENGTH]
        tokens[index, :len(selected)] = torch.tensor([vocabulary.get(word, unknown_id) for word in selected])
    return TensorDataset(tokens, labels)

class TextRCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.LSTM(DIMENSION,RNN_HIDDEN,batch_first=True,bidirectional=True)
        self.project = nn.Linear(DIMENSION+2*RNN_HIDDEN,CHANNELS)
        self.output = nn.Linear(CHANNELS,4)
        self.dropout = nn.Dropout(0.3)
    def forward(self,x,mask):
        packed = nn.utils.rnn.pack_padded_sequence(x,mask.sum(1).cpu(),batch_first=True,enforce_sorted=False)
        hidden,_ = self.rnn(packed)
        hidden,_ = nn.utils.rnn.pad_packed_sequence(hidden,batch_first=True,total_length=x.shape[1])
        left = F.pad(hidden[:,:-1,:RNN_HIDDEN],(0,0,1,0))
        right = F.pad(hidden[:,1:,RNN_HIDDEN:],(0,0,0,1))
        features = torch.tanh(self.project(torch.cat([left,x,right],-1)))
        features = self.dropout(features) # Apply dropout.
        return self.output(features.masked_fill(~mask[:,:,None],-1e9).amax(1))

def label_of(rating):
    assert 1 <= rating <= 10
    return 0 if rating <= 3 else 1 if rating <= 6 else 2 if rating <= 8 else 3

def load_inputs(directory):
    manifest=json.loads((directory/'manifest.json').read_text(encoding='utf-8'))
    if manifest['pipeline']!='latest_clean_data_64_16_20_v2':
        raise ValueError('Use the data exported by 00_prepare_data.ipynb.')
    splits, ids = {}, {}
    for split in ('train','validation'):
        info=manifest['split_info'][split]
        path=directory/info['file']
        if hashlib.sha256(path.read_bytes()).hexdigest()!=info['sha256']:
            raise ValueError('Input hash mismatch: '+split)
        records=[json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()]
        assert len(records)==info['n'] and records
        splits[split]=[]; ids[split]=[]
        for row in records:
            words=row['filtered_words']
            assert isinstance(words,list) and all(isinstance(w,str) and w for w in words)
            # Keep empty reviews in the same partition with an explicit model-input placeholder.
            splits[split].append((words or ['<empty>'],label_of(row['rating'])))
            ids[split].append(row['row_id'])
        assert set(y for _,y in splits[split])==set(range(4)), 'A partition is missing a class.'
    if any(manifest['normalized_text_group_overlap'].values()):
        print('NOTICE: supplied random split has shared normalized text groups:',manifest['normalized_text_group_overlap'])
    return splits,ids,manifest

def fit_vectors(data,manifest,cache_dir,epochs):
    # Cache key binds to the NEW data and every embedding-training setting.
    signature=dict(train=manifest['split_info']['train']['sha256'],validation=manifest['split_info']['validation']['sha256'],
                   method=EMBEDDING_METHOD,seed=SEED,dim=DIMENSION,window=WINDOW,buckets=BUCKETS,
                   min_count=MIN_COUNT,max_vocab=MAX_VOCAB,epochs=epochs,implementation='standalone_v2')
    digest=hashlib.sha256(json.dumps(signature,sort_keys=True).encode()).hexdigest()
    cache_dir.mkdir(parents=True,exist_ok=True)
    path=cache_dir/(EMBEDDING_METHOD+'_'+digest+'.pt')
    if path.exists():
        result=torch.load(path,weights_only=True)
        assert result['signature']==signature
        return result
    frequency=Counter(w for words,_ in data['train'] for w in words)
    words=[w for w,n in frequency.most_common() if n>=MIN_COUNT][:MAX_VOCAB]
    if not words: raise ValueError('Insufficient training vocabulary.')
    vocab={w:i for i,w in enumerate(words,1)}
    pairs,counts=make_pairs([[vocab.get(w,0) for w in row] for row,_ in data['train']])
    if pairs.numel()==0: raise ValueError('No training co-occurrence pairs.')
    oov={}
    if EMBEDDING_METHOD=='fasttext':
        # These strings are only encoded with training-learned subword weights, never fitted on validation.
        oov=dict.fromkeys(sorted({w for rows in data.values() for row,_ in rows for w in row[:MAX_LENGTH] if w not in vocab}))
    vectors,subword_source,used_subwords=train_embedding(EMBEDDING_METHOD,words,pairs,counts,
                            torch.tensor([0,*[frequency[w] for w in words]],dtype=torch.float32),oov,epochs)
    unknown=len(words)+1
    vectors=torch.cat([vectors,torch.zeros(1,DIMENSION)])
    if oov:
        vocab.update((word,unknown+1+i) for i,word in enumerate(oov))
        vectors=torch.cat([vectors,F.normalize(torch.stack(list(oov.values())),dim=1)])
    result=dict(vectors=vectors,vocab=vocab,unknown=unknown,signature=signature,subword_source=subword_source,used_subwords=used_subwords)
    temporary=path.with_suffix('.tmp')
    torch.save(result,temporary); temporary.replace(path)
    return result

def evaluate(model,embedding,loader):
    model.eval()
    probability=[]; truth=[]
    with torch.no_grad():
        for ids,y in loader:
            probability.append(model(embedding(ids),ids.ne(0)).softmax(1).numpy())
            truth.extend(y.tolist())
    target=np.asarray(truth); probability=np.concatenate(probability)
    predicted=probability.argmax(1)
    report=classification_report(target,predicted,labels=range(4),target_names=NAMES,output_dict=True,zero_division=0)
    metrics=dict(accuracy=float((target==predicted).mean()),macro_f1=report['macro avg']['f1-score'],
                 precision_macro=report['macro avg']['precision'],recall_macro=report['macro avg']['recall'],
                 classification_report=report,confusion_matrix=confusion_matrix(target,predicted,labels=range(4)).tolist())
    return metrics,target,probability

def main(argv=None):
    parser=argparse.ArgumentParser()
    parser.add_argument('--data-dir',type=Path,default=PACKAGE_DIR/'data_64_16_20')
    parser.add_argument('--output-root',type=Path,default=PACKAGE_DIR/'results')
    parser.add_argument('--epochs',type=int,default=EPOCHS)
    parser.add_argument('--embedding-epochs',type=int,default=EMBEDDING_EPOCHS)
    parser.add_argument('--weight-alpha',type=float,choices=[0.0,0.5,1.0],default=WEIGHT_ALPHA,
                        help='0=unweighted; 0.5=square-root; 1=inverse. MUST be identical in all six combinations.')
    args=parser.parse_args(argv)
    assert args.epochs>0 and args.embedding_epochs>0
    assert KERNEL_SIZE>0 and KERNEL_SIZE%2==1 and BLOCKS>0 and RNN_HIDDEN>0
    torch.set_num_threads(4)
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    data,row_ids,manifest=load_inputs(args.data_dir)
    cached=fit_vectors(data,manifest,args.data_dir/'embedding_cache',args.embedding_epochs)
    encoded={s:encode(rows,cached['vocab'],cached['unknown']) for s,rows in data.items()}
    torch.manual_seed(SEED)
    model=TextRCNN()
    embedding=nn.Embedding.from_pretrained(cached['vectors'],freeze=True,padding_idx=0)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    generator=torch.Generator().manual_seed(SEED)
    training=DataLoader(encoded['train'],batch_size=BATCH_SIZE,shuffle=True,generator=generator)
    validation=DataLoader(encoded['validation'],batch_size=256)
    counts=torch.bincount(encoded['train'].tensors[1],minlength=4).float()
    weights=(counts.sum()/(4*counts)).pow(args.weight_alpha); weights/=weights.mean()
    run=args.output_root/(MODEL_NAME+'_'+EMBEDDING_METHOD+('_sqrt' if args.weight_alpha==0.5 else '_unweighted' if args.weight_alpha==0 else '_inverse'))/(str(time.time_ns()))
    run.mkdir(parents=True,exist_ok=False)
    config=dict(model=MODEL_NAME,embedding=EMBEDDING_METHOD,labels=dict(zip(NAMES,[[1,2,3],[4,5,6],[7,8],[9,10]])),
                input_manifest=manifest,seed=SEED,epochs=args.epochs,embedding_epochs=args.embedding_epochs,
                train_n=len(data['train']),validation_n=len(data['validation']),class_counts=counts.tolist(),
                embedding_dimension=DIMENSION,max_tokens=MAX_LENGTH,embedding_frozen=True,batch_size=BATCH_SIZE,
                learning_rate=LEARNING_RATE,weight_decay=WEIGHT_DECAY,gradient_clip=GRADIENT_CLIP,weight_alpha=args.weight_alpha,weights=weights.tolist(),
                architecture=dict(DIMENSION=DIMENSION,CHANNELS=CHANNELS,BLOCKS=BLOCKS,KERNEL_SIZE=KERNEL_SIZE,RNN_HIDDEN=RNN_HIDDEN,BUCKETS=BUCKETS),selection='best validation macro_f1',independent_test_evaluated=False,torch_version=str(torch.__version__))
    (run/'config.json').write_text(json.dumps(config,indent=2,ensure_ascii=False),encoding='utf-8')
    history=[]; best=-1
    for epoch in range(1,args.epochs+1):
        model.train(); numerator=denominator=0.; start=time.perf_counter()
        for ids,y in training:
            optimizer.zero_grad()
            loss=F.cross_entropy(model(embedding(ids),ids.ne(0)),y,weight=weights)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),GRADIENT_CLIP); optimizer.step()
            mass=weights[y].sum().item(); numerator+=loss.item()*mass; denominator+=mass
        metrics,target,probability=evaluate(model,embedding,validation)
        row=dict(epoch=epoch,train_loss=numerator/denominator,seconds=time.perf_counter()-start,**metrics)
        history.append(row)
        if row['macro_f1']>best:
            best=row['macro_f1']
            torch.save(dict(model_state=model.state_dict(),vectors=cached['vectors'],vocab=cached['vocab'],
                            unknown=cached['unknown'],config=config,subword_source=cached['subword_source'],used_subwords=cached['used_subwords']),run/'best_model.pt')
            np.savez_compressed(run/'validation_predictions.npz',row_id=np.asarray(row_ids['validation']),target=target,probability=probability)
        (run/'history.json').write_text(json.dumps(history,indent=2),encoding='utf-8')
        print(MODEL_NAME,EMBEDDING_METHOD,'epoch',epoch,{k:round(row[k],4) for k in ('accuracy','macro_f1','precision_macro','recall_macro')},flush=True)
    selected=max(history,key=lambda r:r['macro_f1'])
    checkpoint=torch.load(run/'best_model.pt',weights_only=True)
    model.load_state_dict(checkpoint['model_state'])
    verified,_,_=evaluate(model,embedding,validation)
    assert all(np.isclose(verified[k],selected[k],atol=1e-8,rtol=0) for k in ('accuracy','macro_f1','precision_macro','recall_macro'))
    (run/'best_metrics.json').write_text(json.dumps(selected,indent=2),encoding='utf-8')
    print('Verified results saved to',run.resolve())
    return run


def evaluate_final_test(run_directory, data_directory):
    # Call only after all tuning choices have been finalized.
    checkpoint=torch.load(Path(run_directory)/'best_model.pt',weights_only=True)
    cfg=checkpoint['config']
    if cfg['model']!=MODEL_NAME or cfg['embedding']!=EMBEDDING_METHOD:
        raise ValueError('Checkpoint belongs to a different combination.')
    globals().update(cfg['architecture'])
    globals()['MAX_LENGTH']=cfg['max_tokens']
    manifest=json.loads((Path(data_directory)/'manifest.json').read_text(encoding='utf-8'))
    if manifest!=cfg['input_manifest']: raise ValueError('Checkpoint and data manifest differ.')
    info=manifest['split_info']['test']; path=Path(data_directory)/info['file']
    if hashlib.sha256(path.read_bytes()).hexdigest()!=info['sha256']: raise ValueError('Test hash mismatch.')
    records=[json.loads(line) for line in path.read_text(encoding='utf-8').splitlines()]
    assert len(records)==info['n'] and records
    rows=[(r['filtered_words'] or ['<empty>'],label_of(r['rating'])) for r in records]
    vocab=dict(checkpoint['vocab']); vectors=checkpoint['vectors']
    if EMBEDDING_METHOD=='fasttext':
        # Encode unseen TEST words using stored training-learned subword weights only.
        source=checkpoint['subword_source']; used=set(checkpoint['used_subwords'])
        unseen=sorted({w for words,_ in rows for w in words[:MAX_LENGTH] if w not in vocab})
        additions=[]
        for word in unseen:
            pieces=[p for p in character_ngrams(word) if p in used]
            value=source[pieces].mean(0) if pieces else torch.zeros(DIMENSION)
            vocab[word]=len(vectors)+len(additions); additions.append(value)
        if additions: vectors=torch.cat([vectors,F.normalize(torch.stack(additions),dim=1)])
    model=TextRCNN(); model.load_state_dict(checkpoint['model_state'])
    embedding=nn.Embedding.from_pretrained(vectors,freeze=True,padding_idx=0)
    loader=DataLoader(encode(rows,vocab,checkpoint['unknown']),batch_size=256)
    metrics,target,probability=evaluate(model,embedding,loader)
    destination=Path(run_directory)/'final_test_metrics.json'
    destination.write_text(json.dumps(metrics,indent=2),encoding='utf-8')
    np.savez_compressed(Path(run_directory)/'final_test_predictions.npz',row_id=np.asarray([r['row_id'] for r in records]),target=target,probability=probability)
    print('FINAL TEST:', {k:metrics[k] for k in ('accuracy','macro_f1','precision_macro','recall_macro')})
    return metrics


In [ ]:
from pathlib import Path

PACKAGE_DIR = Path.cwd()
DATA_DIR = PACKAGE_DIR / 'data_64_16_20'

print("DATA_DIR =", DATA_DIR)
print("exists?", DATA_DIR.exists())

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Data directory not found: {DATA_DIR}")

run_directory = main(['--data-dir', str(DATA_DIR)])


In [ ]:
# Keep this False during tuning. Set it to True after fixing the configuration to evaluate the saved checkpoint.
RUN_FINAL_TEST = True
if RUN_FINAL_TEST:
    final_test_results = evaluate_final_test(run_directory, DATA_DIR)
